In [7]:
import pandas as pd
import time
import torch
import umap.umap_ as umap
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer


In [ ]:

# ==========================================
# 0. DÉTECTION DU MATÉRIEL
# ==========================================
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Accélération matérielle activée sur : {device.upper()}\n")

print("Chargement du modèle BERT (all-MiniLM-L6-v2)...")
modele_bert = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. PRÉPARATION DES DONNÉES (PARTIE 1 : CSV)
# ==========================================
print("\n1. Chargement des données CSV...")
df = pd.read_csv('C:/Users/titou/Documents/dev/Ynov/DataScience/Mini-memoire-Base-vectorielles/Data/train_tm/train.csv')

# df = df.sample(3000, random_state=42) # Pour tester rapidement

df['TEXTE_COMPLET'] = df['TITLE'] + " " + df['ABSTRACT']
X_textes = df['TEXTE_COMPLET'].tolist()
labels_cols = ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']
y_labels = df[labels_cols].values

X_train_text, X_test_text, y_train, y_test = train_test_split(X_textes, y_labels, test_size=0.2, random_state=42)
clf_multi = OneVsRestClassifier(LogisticRegression(max_iter=1000))


In [ ]:

# --- EXPÉRIENCE A : TF-IDF ---
print("\n--- [1/3] Exécution de TF-IDF ---")
t0 = time.time()
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)
clf_multi.fit(X_train_tfidf, y_train)
score_tfidf = f1_score(y_test, clf_multi.predict(X_test_tfidf), average='micro')
print(f"TF-IDF terminé | Score F1: {score_tfidf:.4f} | Temps: {time.time() - t0:.2f}s")


In [ ]:

# --- EXPÉRIENCE B : BERT BASELINE ---
print("\n--- [2/3] Exécution de BERT (384 dim) ---")
t0 = time.time()
X_train_bert_p1 = modele_bert.encode(X_train_text, batch_size=128, show_progress_bar=True)
X_test_bert_p1 = modele_bert.encode(X_test_text, batch_size=128, show_progress_bar=False)
clf_multi.fit(X_train_bert_p1, y_train)
score_bert_p1 = f1_score(y_test, clf_multi.predict(X_test_bert_p1), average='micro')
print(f"BERT terminé | Score F1: {score_bert_p1:.4f} | Temps: {time.time() - t0:.2f}s")

# --- EXPÉRIENCE C : BERT + PCA ---
print("\n--- [3/3] Exécution de BERT + PCA ---")
t0 = time.time()
pca = PCA(n_components=0.95, random_state=42)
X_train_bert_pca = pca.fit_transform(X_train_bert_p1)
X_test_bert_pca = pca.transform(X_test_bert_p1)
clf_multi.fit(X_train_bert_pca, y_train)
score_pca_p1 = f1_score(y_test, clf_multi.predict(X_test_bert_pca), average='micro')
print(f"BERT+PCA terminé | Dimensions: 384 -> {X_train_bert_pca.shape[1]} | Score F1: {score_pca_p1:.4f} | Temps: {time.time() - t0:.2f}s\n")



In [ ]:

# =========================================================
# 2. PRÉPARATION DES DONNÉES (PARTIE 2 : 20 NEWSGROUPS)
# =========================================================
print("="*65)
print("2. Chargement des données (Espace vs Médecine)...")
categories = ['sci.med', 'sci.space']
dataset = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

textes = dataset.data[:1000]
labels = dataset.target[:1000]
X_train_text_n, X_test_text_n, y_train_n, y_test_n = train_test_split(textes, labels, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)


In [ ]:

# =========================================================
# ENCODAGE UNIQUE AVEC BERT MINILM
# =========================================================
print("\n--- Encodage Global avec BERT ---")
t0 = time.time()
X_train_bert_p2 = modele_bert.encode(X_train_text_n, batch_size=128, show_progress_bar=True)
X_test_bert_p2 = modele_bert.encode(X_test_text_n, batch_size=128, show_progress_bar=False)
t_encodage_global = time.time() - t0
print(f"Encodage terminé en {t_encodage_global:.2f}s.")



In [ ]:


# ---------------------------------------------------------
# EXPÉRIENCES COMPARATIVES (Démonstration pour le mémoire)
# ---------------------------------------------------------
print("\n--- [1/4] Baseline (384 dim) ---")
clf.fit(X_train_bert_p2, y_train_n)
score_base = f1_score(y_test_n, clf.predict(X_test_bert_p2), average='weighted')
print(f"Baseline terminée | Score F1: {score_base:.4f}")



In [ ]:


print("\n--- [2/4] PCA (64 dim) ---")
t0 = time.time()
pca_n = PCA(n_components=64, random_state=42)
X_train_pca_n = pca_n.fit_transform(X_train_bert_p2)
X_test_pca_n = pca_n.transform(X_test_bert_p2)
t_pca_n = time.time() - t0
clf.fit(X_train_pca_n, y_train_n)
score_pca_n = f1_score(y_test_n, clf.predict(X_test_pca_n), average='weighted')
print(f"PCA terminée | Score F1: {score_pca_n:.4f} | Temps de réduction: {t_pca_n:.4f}s")



In [ ]:


print("\n--- [3/4] UMAP (64 dim) ---")
t0 = time.time()
reducer = umap.UMAP(n_components=64, random_state=42)
X_train_umap = reducer.fit_transform(X_train_bert_p2)
X_test_umap = reducer.transform(X_test_bert_p2)
t_umap = time.time() - t0
clf.fit(X_train_umap, y_train_n)
score_umap = f1_score(y_test_n, clf.predict(X_test_umap), average='weighted')
print(f"UMAP terminé | Score F1: {score_umap:.4f} | Temps de réduction: {t_umap:.4f}s")



In [ ]:


print("\n--- [4/4] Troncature directe type Matryoshka (64 dim) ---")
print("ATTENTION : BERT n'étant pas entraîné en MRL, on s'attend à une chute du score F1.")
t0 = time.time()
# Coupe brutale des 384 dimensions vers 64 dimensions
X_train_fake_mrl = X_train_bert_p2[:, :64]
X_test_fake_mrl = X_test_bert_p2[:, :64]
t_fake_mrl = time.time() - t0 

clf.fit(X_train_fake_mrl, y_train_n)
score_fake_mrl = f1_score(y_test_n, clf.predict(X_test_fake_mrl), average='weighted')
print(f"Troncature terminée | Score F1: {score_fake_mrl:.4f} | Temps de réduction: {t_fake_mrl:.6f}s (Immédiat)\n")